# Asignación de atributos de TransCAD a las vialidades principales de OSM

Esta notebook asigna los atributos viales de TransCAD hacia la geometría de OpenStreetMap mediante relaciones espaciales, de orientación y distancia geométrica.

El proceso nos sirve para caracterizar las vialidades principales, propagar atributos por nombre, identificar *links*, cuchillas y laterales, y completr los valores faltantes mediante medianas por tipo de vialidad.

In [1]:
from pathlib import Path
import ast
import geopandas as gpd
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 150)

### Lectura y preparación de las redes viales

Cargamos las redes de TransCAD y OSM; reproyectamos OSM al sistema de coordenadas de TransCAD para que las distancias y los buffers se calculen en metros.

In [ ]:
ruta_datos = Path("/Users/.../datos_asignacion_atributos")
datos_transcad = gpd.read_file(ruta_datos / "AMG_RedVial_2023_con_limites_velocidad.shp")
datos_osm = gpd.read_file(ruta_datos / "edges_visum.shp")
datos_osm_metricos = datos_osm.to_crs(datos_transcad.crs)

columnas_atributos = ["CAPACIDAD", "CARRILES", "Velocidad_", "Limite_vel"]

for col in columnas_atributos:
    datos_transcad[col] = pd.to_numeric(datos_transcad[col], errors="coerce")

### Selección de vialidades principales

Consideramos como vialidades principales de TransCAD aquellas cuyo límite de velocidad es mayor o igual a 40 km/h.

También creamos en OSM las columnas que recibirán los cuatro atributos transferidos.

In [3]:
# Seleccionamos las vialidades principales de TransCAD
transcad_principales = datos_transcad[datos_transcad["Limite_vel"] >= 40].copy()

# Creamos la red OSM de trabajo y las columnas que recibirán los atributos
osm_trabajo = datos_osm_metricos.copy()
osm_trabajo["cap_final"] = np.nan
osm_trabajo["carriles_final"] = np.nan
osm_trabajo["velprom_final"] = np.nan
osm_trabajo["vel_final"] = np.nan

columnas_finales = ["cap_final", "carriles_final", "velprom_final", "vel_final"]

print(f"Vialidades principales TransCAD: {len(transcad_principales):,}")
print(f"Links OSM totales: {len(osm_trabajo):,}")

Vialidades principales TransCAD: 7,734
Links OSM totales: 494,896


### Orientación geométrica de los links

La orientación la calculamos con los extremos de cada `LineString`. El ángulo se expresa entre 0° y 180°, lo que implica que dos segmentos contrarios que forman parte del mismo eje vial, conservan la misma orientación.

In [4]:
# Calculamos la orientación principal de cada segmento
def calcular_orientacion(geom):
    if geom is None or geom.is_empty:
        return np.nan

    coords = list(geom.coords)

    if len(coords) < 2:
        return np.nan

    x1, y1 = coords[0]
    x2, y2 = coords[-1]
    dx = x2 - x1
    dy = y2 - y1
    angulo = np.degrees(np.arctan2(dy, dx))

    return angulo % 180

transcad_principales["orientacion"] = transcad_principales.geometry.apply(calcular_orientacion)
osm_trabajo["orientacion"] = osm_trabajo.geometry.apply(calcular_orientacion)

print(f"Orientaciones válidas TransCAD: {transcad_principales['orientacion'].notna().sum():,} / {len(transcad_principales):,}")
print(f"Orientaciones válidas OSM: {osm_trabajo['orientacion'].notna().sum():,} / {len(osm_trabajo):,}")

Orientaciones válidas TransCAD: 7,734 / 7,734
Orientaciones válidas OSM: 494,896 / 494,896


### Generación de candidatos espaciales

Para cada link OSM generamos un buffer de 25 metros. Los segmentos de TransCAD que intersectan ese buffer se consideran candidatos potenciales.

In [5]:
buffer_m = 25
angulo_maximo = 35

red_destino = osm_trabajo.copy().reset_index(drop=True)  # OSM = red destino
red_base = transcad_principales.copy().reset_index(drop=True)  # TransCAD = red base

red_destino["_idx_osm"] = red_destino.index
red_base["_idx_transcad"] = red_base.index

# Creamos un buffer alrededor de cada link OSM
red_destino_buffer = red_destino[["_idx_osm", "orientacion", "geometry"]].copy()
red_destino_buffer["geometry"] = red_destino_buffer.geometry.buffer(buffer_m)
red_destino_buffer = red_destino_buffer.rename(columns={"orientacion": "orientacion_osm"})

# Buscamos los links TransCAD que intersectan cada buffer OSM
candidatos = gpd.sjoin(red_base[["_idx_transcad", "orientacion", "geometry"] + columnas_atributos], red_destino_buffer[["_idx_osm", "orientacion_osm", "geometry"]], how="inner", predicate="intersects")
candidatos = candidatos.rename(columns={"orientacion": "orientacion_transcad"})

print(f"Candidatos espaciales brutos: {len(candidatos):,}")
print(f"Links OSM únicos con al menos un candidato: {candidatos['_idx_osm'].nunique():,}")

Candidatos espaciales brutos: 162,171
Links OSM únicos con al menos un candidato: 70,709


### Filtro por orientación

El filtro angular nos elimina correspondencias entre segmentos que se encuentran cerca, pero que representan vialidades diferentes.

Nos quedamos con el candidato cuando la diferencia entre las orientaciones de OSM y TransCAD es menor a 35°.

In [6]:
# Calculamos la diferencia angular mínima considerando la equivalencia entre 0° y 180°
def diferencia_angular(a, b):
    diferencia = abs(a - b)
    return min(diferencia, 180 - diferencia)


candidatos["dif_angular"] = candidatos.apply(lambda fila: diferencia_angular(fila["orientacion_transcad"], fila["orientacion_osm"]), axis=1)
candidatos_filtrados = candidatos[candidatos["dif_angular"] < angulo_maximo].copy()

print(f"Candidatos después del filtro angular: {len(candidatos_filtrados):,}")
print(f"Links OSM únicos después del filtro angular: {candidatos_filtrados['_idx_osm'].nunique():,}")

Candidatos después del filtro angular: 88,646
Links OSM únicos después del filtro angular: 37,361


### Selección del mejor *match* directo

Para cada candidato calculamos la distancia real entre la geometría original de OSM y la geometría original de TransCAD. El mejor match se selecciona primero por menor distancia y, en caso de empate, por menor diferencia angular.

In [7]:
# Calculamos la distancia real entre cada par de geometrías candidatas
candidatos_filtrados["distancia_m"] = candidatos_filtrados.apply(lambda fila: red_base.loc[fila["_idx_transcad"], "geometry"].distance(red_destino.loc[fila["_idx_osm"], "geometry"]), axis=1)

# Seleccionamos un único segmento TransCAD para cada link OSM
mejor_match = candidatos_filtrados.sort_values(["_idx_osm", "distancia_m", "dif_angular"]).drop_duplicates(subset="_idx_osm", keep="first").copy()

print(f"Links OSM con match directo: {len(mejor_match):,}")
print(f"Mediana de distancia: {mejor_match['distancia_m'].median():.2f} m")
print(f"Mediana de diferencia angular: {mejor_match['dif_angular'].median():.2f}°")

Links OSM con match directo: 37,361
Mediana de distancia: 5.45 m
Mediana de diferencia angular: 1.34°


### Transferencia de atributos

Transferimos al *link* OSM los atributos del segmento TransCAD seleccionado como mejor correspondencia:

- Capacidad vial
- Número de carriles
- Velocidad promedio de operación
- Límite de velocidad

In [8]:
# Transferimos los atributos del mejor match hacia OSM
osm_trabajo.loc[mejor_match["_idx_osm"], "cap_final"] = mejor_match["CAPACIDAD"].values
osm_trabajo.loc[mejor_match["_idx_osm"], "carriles_final"] = mejor_match["CARRILES"].values
osm_trabajo.loc[mejor_match["_idx_osm"], "velprom_final"] = mejor_match["Velocidad_"].values
osm_trabajo.loc[mejor_match["_idx_osm"], "vel_final"] = mejor_match["Limite_vel"].values

print("Atributos asignados mediante match directo:")
print(osm_trabajo[columnas_finales].notna().sum())

Atributos asignados mediante match directo:
cap_final         37361
carriles_final    37361
velprom_final     37361
vel_final         37361
dtype: int64


### Exclusión de vialidades no objetivo

El match puede asignar atributos a vialidades cercanas que no pertenecen al conjunto principal. Por ello, normalizamos el atributo `highway` y eliminamos cualquier asignación realizada sobre vialidades peatonales, ciclistas, residenciales, de servicio o de jerarquía no principal.

In [9]:
# Tipos de vialidad que no deben conservar atributos en esta notebook
tipos_prohibidos = {"unclassified", "residential", "living_street", "pedestrian", "service", "steps", "footway", "path", "cycleway", "track", "bridleway", "corridor"}

# Normalizamos highway cuando contiene un valor simple o una colección de valores
def obtener_tipos_highway(valor):
    if pd.isna(valor):
        return []

    if isinstance(valor, (list, tuple, set)):
        return [str(tipo).strip() for tipo in valor]

    if isinstance(valor, str):
        try:
            valor_lista = ast.literal_eval(valor)

            if isinstance(valor_lista, (list, tuple, set)):
                return [str(tipo).strip() for tipo in valor_lista]

        except (ValueError, SyntaxError):
            pass

        return [valor.strip()]

    return [str(valor).strip()]


# Identificamos links que contienen al menos un tipo prohibido
def contiene_tipo_prohibido(valor):
    tipos = obtener_tipos_highway(valor)
    return any(tipo in tipos_prohibidos for tipo in tipos)


mascara_prohibidos = osm_trabajo["highway"].apply(contiene_tipo_prohibido)
osm_trabajo.loc[mascara_prohibidos, columnas_finales] = np.nan

### Propagación por nombre de vialidad

Los *links* de una misma vialidad pueden estar divididos en múltiples segmentos. Cuando algunos segmentos ya tienen atributos y otros no, para los atributos continuos, usamos la mediana de los *links* con el mismo nombre para completar los faltantes, mientras que para atributos discretos usamos una moda conservadora para completar los faltantes. 

In [10]:
def moda_conservadora(serie):
    modas = serie.dropna().mode()
    return modas.min() if not modas.empty else np.nan

In [11]:
# Calculamos las medianas de los atributos para cada nombre de vialidad
medianas_por_nombre = osm_trabajo[(~mascara_prohibidos) & osm_trabajo["name"].notna()].groupby("name").agg(cap_final=("cap_final", "median"), carriles_final=("carriles_final", moda_conservadora), velprom_final=("velprom_final", "median"), vel_final=("vel_final", "median"))

# Propagamos las medianas a links faltantes con el mismo nombre
for col in columnas_finales:
    mascara_rellenar = ~mascara_prohibidos & osm_trabajo[col].isna() & osm_trabajo["name"].isin(medianas_por_nombre.index)
    osm_trabajo.loc[mascara_rellenar, col] = osm_trabajo.loc[mascara_rellenar, "name"].map(medianas_por_nombre[col])

print("Atributos disponibles después de la propagación por nombre:")
print(osm_trabajo[columnas_finales].notna().sum())

Atributos disponibles después de la propagación por nombre:
cap_final         29376
carriles_final    29376
velprom_final     29376
vel_final         29376
dtype: int64


### Identificación de conectores o cuchillas

Los conectores los trabajamos de manera independiente porque suelen ser segmentos cortos, curvos y separados de los ejes principales.

En TransCAD vienen identificados como `TIPO == 1`, mientras que en OSM corresponden a categorías `highway` terminadas en `_link`.

In [12]:
# Seleccionamos los conectores de TransCAD
transcad_cuchillas = datos_transcad[datos_transcad["TIPO"] == 1].copy()

# Identificamos conectores OSM mediante las categorías terminadas en "_link"
def es_cuchilla(valor):
    tipos = obtener_tipos_highway(valor)
    return any(tipo.endswith("_link") for tipo in tipos)


mascara_cuchillas = osm_trabajo["highway"].apply(es_cuchilla)

transcad_cuchillas["orientacion"] = transcad_cuchillas.geometry.apply(calcular_orientacion)
osm_cuchillas = osm_trabajo[mascara_cuchillas].copy()
osm_cuchillas["_idx_osm"] = osm_cuchillas.index

# Creamos buffers alrededor de las cuchillas OSM
osm_cuchillas_buffer = osm_cuchillas[["_idx_osm", "orientacion", "geometry"]].copy()
osm_cuchillas_buffer["geometry"] = osm_cuchillas_buffer.geometry.buffer(buffer_m)
osm_cuchillas_buffer = osm_cuchillas_buffer.rename(columns={"orientacion": "orientacion_osm"})

# Creamos un índice interno para los conectores TransCAD
transcad_cuchillas = transcad_cuchillas.reset_index(drop=True)
transcad_cuchillas["_idx_transcad"] = transcad_cuchillas.index

# Generamos candidatos espaciales
candidatos_cuchillas = gpd.sjoin(transcad_cuchillas[["_idx_transcad", "orientacion", "geometry"] + columnas_atributos], osm_cuchillas_buffer[["_idx_osm", "orientacion_osm", "geometry"]], how="inner", predicate="intersects")
candidatos_cuchillas = candidatos_cuchillas.rename(columns={"orientacion": "orientacion_transcad"})

Aplicamos a los conectores los mismos criterios utilizados en el *match* principal: diferencia angular menor a 35°, menor distancia geométrica y diferencia angular como criterio de desempate.

In [13]:
# Filtramos los candidatos por orientación
candidatos_cuchillas["dif_angular"] = candidatos_cuchillas.apply(lambda fila: diferencia_angular(fila["orientacion_transcad"], fila["orientacion_osm"]), axis=1)
candidatos_cuchillas = candidatos_cuchillas[candidatos_cuchillas["dif_angular"] < angulo_maximo].copy()

# Calculamos la distancia geométrica real
candidatos_cuchillas["distancia_m"] = candidatos_cuchillas.apply(lambda fila: transcad_cuchillas.loc[fila["_idx_transcad"], "geometry"].distance(osm_trabajo.loc[fila["_idx_osm"], "geometry"]), axis=1)

# Seleccionamos el mejor candidato para cada cuchilla OSM
mejor_match_cuchillas = candidatos_cuchillas.sort_values(["_idx_osm", "distancia_m", "dif_angular"]).drop_duplicates(subset="_idx_osm", keep="first").copy()

# Transferimos los atributos
osm_trabajo.loc[mejor_match_cuchillas["_idx_osm"], "cap_final"] = mejor_match_cuchillas["CAPACIDAD"].values
osm_trabajo.loc[mejor_match_cuchillas["_idx_osm"], "carriles_final"] = mejor_match_cuchillas["CARRILES"].values
osm_trabajo.loc[mejor_match_cuchillas["_idx_osm"], "velprom_final"] = mejor_match_cuchillas["Velocidad_"].values
osm_trabajo.loc[mejor_match_cuchillas["_idx_osm"], "vel_final"] = mejor_match_cuchillas["Limite_vel"].values

print(f"Cuchillas OSM totales: {mascara_cuchillas.sum():,}")
print(f"Cuchillas OSM con match TransCAD: {len(mejor_match_cuchillas):,}")

Cuchillas OSM totales: 3,082
Cuchillas OSM con match TransCAD: 639


### Identificación de laterales

Las laterales de TransCAD vienen identificadas como `TIPO == 8`, y también se procesan de forma independiente.

Los candidatos se buscan contra los buffers de toda la red OSM, conservando los mismos filtros geométricos del procedimiento anterior.

In [14]:
# Seleccionamos las vialidades laterales de TransCAD
transcad_laterales = datos_transcad[datos_transcad["TIPO"] == 8].copy()
transcad_laterales["orientacion"] = transcad_laterales.geometry.apply(calcular_orientacion)
transcad_laterales = transcad_laterales.reset_index(drop=True)
transcad_laterales["_idx_transcad"] = transcad_laterales.index

# Generamos candidatos espaciales
candidatos_laterales = gpd.sjoin(transcad_laterales[["_idx_transcad", "orientacion", "geometry"] + columnas_atributos], red_destino_buffer[["_idx_osm", "orientacion_osm", "geometry"]], how="inner", predicate="intersects")
candidatos_laterales = candidatos_laterales.rename(columns={"orientacion": "orientacion_transcad"})

# Aplicamos el filtro angular
candidatos_laterales["dif_angular"] = candidatos_laterales.apply(lambda fila: diferencia_angular(fila["orientacion_transcad"], fila["orientacion_osm"]), axis=1)
candidatos_laterales = candidatos_laterales[candidatos_laterales["dif_angular"] < angulo_maximo].copy()

# Calculamos la distancia real entre cada par candidato
candidatos_laterales["distancia_m"] = candidatos_laterales.apply(lambda fila: transcad_laterales.loc[fila["_idx_transcad"], "geometry"].distance(osm_trabajo.loc[fila["_idx_osm"], "geometry"]), axis=1)

# Seleccionamos el mejor candidato por link OSM
mejor_match_laterales = candidatos_laterales.sort_values(["_idx_osm", "distancia_m", "dif_angular"]).drop_duplicates(subset="_idx_osm", keep="first").copy()

# Transferimos los atributos
osm_trabajo.loc[mejor_match_laterales["_idx_osm"], "cap_final"] = mejor_match_laterales["CAPACIDAD"].values
osm_trabajo.loc[mejor_match_laterales["_idx_osm"], "carriles_final"] = mejor_match_laterales["CARRILES"].values
osm_trabajo.loc[mejor_match_laterales["_idx_osm"], "velprom_final"] = mejor_match_laterales["Velocidad_"].values
osm_trabajo.loc[mejor_match_laterales["_idx_osm"], "vel_final"] = mejor_match_laterales["Limite_vel"].values

print(f"Laterales TransCAD: {len(transcad_laterales):,}")
print(f"Links OSM identificados como laterales: {len(mejor_match_laterales):,}")

Laterales TransCAD: 1,601
Links OSM identificados como laterales: 8,255


### Clasificación final de las vialidades objetivo

Después de transferir conectores y laterales, volvemos a limpiar las vialidades invalidas.

Posteriormente, clasificamos las vialidades principales de OSM mediante su categoría `highway`. Esta será utilizada para completar los atributos que continúen faltantes.

In [15]:
# Eliminamos nuevamente cualquier atributo asignado a una vialidad prohibida
mascara_prohibidos = osm_trabajo["highway"].apply(contiene_tipo_prohibido)
osm_trabajo.loc[mascara_prohibidos, columnas_finales] = np.nan

# Tipos OSM que forman parte de la red vial objetivo
tipos_validos = {"motorway", "trunk", "primary", "secondary", "tertiary", "motorway_link", "trunk_link", "primary_link", "secondary_link", "tertiary_link"}

# Obtenemos una categoría highway normalizada para cada link objetivo
def obtener_highway_tipo(valor):
    tipos = obtener_tipos_highway(valor)

    if any(tipo in tipos_prohibidos for tipo in tipos):
        return np.nan

    for tipo in tipos:
        if tipo in tipos_validos:
            return tipo

    return np.nan


osm_trabajo["highway_tipo"] = osm_trabajo["highway"].apply(obtener_highway_tipo)
osm_trabajo["es_vialidad_objetivo"] = osm_trabajo["highway_tipo"].notna()

print(osm_trabajo["highway_tipo"].value_counts(dropna=False))

highway_tipo
NaN               444475
tertiary           27864
secondary          10088
primary             7615
trunk               1414
primary_link         778
tertiary_link        737
trunk_link           554
secondary_link       510
motorway_link        489
motorway             372
Name: count, dtype: int64


### Asignación por tipo de vialidad

Para los *links* objetivo que todavía no tienen atributos, calculamos la mediana o moda conservadora (dependiendo del tipo de atributo) de cada variable dentro de su categoría `highway`; mientras que las vialidades que no pertenecen al conjunto objetivo permanecen sin atributos.

In [16]:
# Calculamos las medianas de cada atributo por tipo de vialidad OSM
medianas_por_tipo = osm_trabajo[osm_trabajo["es_vialidad_objetivo"]].groupby("highway_tipo").agg(cap_final=("cap_final", "median"), carriles_final=("carriles_final", moda_conservadora), velprom_final=("velprom_final", "median"), vel_final=("vel_final", "median"))
display(medianas_por_tipo)

,cap_final,carriles_final,velprom_final,vel_final
highway_tipo,,,,
motorway,4500.0,1.0,34.000021,80.0
motorway_link,3000.0,1.0,24.000000,30.0
primary,4500.0,6.0,25.999972,40.0
primary_link,4000.0,1.0,24.000000,40.0
secondary,4000.0,4.0,24.000040,40.0
secondary_link,4000.0,2.0,24.000057,40.0
tertiary,4000.0,2.0,25.999987,40.0
tertiary_link,4000.0,2.0,24.000000,30.0
trunk,5000.0,2.0,34.000109,80.0


In [17]:
# Completamos los atributos faltantes mediante la mediana de su tipo de vialidad
for col in columnas_finales:
    mascara_rellenar = osm_trabajo["es_vialidad_objetivo"] & osm_trabajo[col].isna()
    osm_trabajo.loc[mascara_rellenar, col] = osm_trabajo.loc[mascara_rellenar, "highway_tipo"].map(medianas_por_tipo[col])

# Nos aseguramos de que las vialidades fuera del objetivo no conserven atributos
osm_trabajo.loc[~osm_trabajo["es_vialidad_objetivo"], columnas_finales] = np.nan

### Preparación y reporte de la capa final

Finalmente, reproyectamos la red al sistema de coordenadas original de OSM y seleccionamos las columnas que formarían parte del archivo de salida.

**Nota importante:** En esta notebook no se genera todavía ningún shapefile o GeoPackage. Únicamente se prepara y se inspecciona el resultado.

In [18]:
# Reproyectamos el resultado al CRS original de OSM
osm_final = osm_trabajo.to_crs(datos_osm.crs).copy()

# Seleccionamos las columnas finales
columnas_salida = ["u", "v", "key", "osmid", "highway", "name", "oneway", "length", "lanes", "maxspeed", "highway_tipo", "es_vialidad_objetivo", "cap_final", "carriles_final", "velprom_final", "vel_final", "geometry"]
columnas_salida = [col for col in columnas_salida if col in osm_final.columns]

osm_final = osm_final[columnas_salida].copy()

In [19]:
# Distribución de atributos por tipo de vialidad
resumen_por_tipo = osm_final[osm_final["es_vialidad_objetivo"]].groupby("highway_tipo").agg(links=("highway_tipo", "size"), capacidad_mediana=("cap_final", "median"), carriles_mediana=("carriles_final", "median"), velocidad_promedio_mediana=("velprom_final", "median"), limite_velocidad_mediana=("vel_final", "median")).sort_values("links", ascending=False)
display(resumen_por_tipo)

,links,capacidad_mediana,carriles_mediana,velocidad_promedio_mediana,limite_velocidad_mediana
highway_tipo,,,,,
tertiary,27864,4000.0,2.0,25.999987,40.0
secondary,10088,4000.0,4.0,24.000040,40.0
primary,7615,4500.0,4.0,25.999972,40.0
trunk,1414,5000.0,2.0,34.000109,80.0
primary_link,778,4000.0,1.0,24.000000,40.0
tertiary_link,737,4000.0,2.0,24.000000,30.0
trunk_link,554,4000.0,2.0,24.000019,30.0
secondary_link,510,4000.0,2.0,24.000057,40.0
motorway_link,489,3000.0,1.0,24.000000,30.0


In [20]:
# Muestra de la capa final preparada
display(osm_final.head())

,u,v,key,osmid,highway,name,oneway,length,lanes,maxspeed,highway_tipo,es_vialidad_objetivo,cap_final,carriles_final,velprom_final,vel_final,geometry
0,267537966,7306651630,0,"[832652768, 619339782, 688424559, 188974544, 1...",motorway,Autopista Guadalajara - Morelia,True,4827.824909,"['3', '2']",80,motorway,True,4000.0,2.0,24.000000,30.0,"LINESTRING (-103.24632 20.61606, -103.24734 20..."
1,267537966,5837556433,0,694566994,motorway_link,None,True,146.028531,1,None,motorway_link,True,4000.0,2.0,23.999956,30.0,"LINESTRING (-103.24632 20.61606, -103.24648 20..."
2,267538751,273140976,0,189118222,motorway,Autopista Guadalajara - Zapotlanejo,True,520.236927,2,None,motorway,True,2500.0,1.0,53.999923,80.0,"LINESTRING (-103.1364 20.60565, -103.13432 20...."
3,267538751,1746293763,0,907206852,motorway_link,Autopista Guadalajara - Morelia,True,617.213211,1,None,motorway_link,True,1500.0,1.0,16.000019,30.0,"LINESTRING (-103.1364 20.60565, -103.1354 20.6..."
4,273140976,1997658287,0,835083267,motorway,Autopista Guadalajara - Zapotlanejo,True,151.355373,3,None,motorway,True,5000.0,2.0,53.999997,80.0,"LINESTRING (-103.13185 20.60758, -103.13162 20..."
